# 4 · Time Series as Features (Lags, Leads, Rolling Stats)

**Cycles** are growth and decay that depend on the recent past, not on the calendar. Model them with **lag features**.

**Process:**
1. **Deseasonalize first.** Trend and seasons also create autocorrelation and would hide the cycles.
2. **Choose the lags** with the PACF (linear) and lag plots (non-linear).
3. **Look for leading indicators**, other series that move before the target. Leads are allowed only if their values are known ahead of time, like planned promotions.
4. **Build, fit, and evaluate** on a time-ordered split.

⚠️ With lag-1 features you can only forecast 1 step ahead. Lesson 6 covers longer horizons.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 4))
plot_params = dict(color="0.75", style=".-", markeredgecolor="0.25", markerfacecolor="0.25", legend=False)

DATA = Path("../input/store-sales-time-series-forecasting")   # change to where you unzipped the data

In [ ]:
def plot_lags(x, y=None, lags=6):
    """Scatter y against x shifted by 1..lags, with the correlation in each title. Negative lags = leads."""
    y = x if y is None else y
    lags = range(1, lags + 1) if isinstance(lags, int) else lags
    fig, axs = plt.subplots(1, len(lags), figsize=(2.2 * len(lags), 2.6), sharey=True)
    for ax, k in zip(axs, lags):
        ax.scatter(x.shift(k), y, s=4, alpha=0.6)
        ax.set_title(f"{'Lag' if k > 0 else 'Lead'} {abs(k)}  r={y.corr(x.shift(k)):.2f}", fontsize=10)
    return fig


def make_lags(ts, lags, lead_time=1, name="y"):
    return pd.concat({f"{name}_lag_{i}": ts.shift(i) for i in range(lead_time, lags + lead_time)}, axis=1)


def make_leads(ts, leads, name="y"):
    return pd.concat({f"{name}_lead_{i}": ts.shift(-i) for i in range(1, leads + 1)}, axis=1)

In [ ]:
store_sales = pd.read_csv(
    DATA / "train.csv",
    usecols=["store_nbr", "family", "date", "sales", "onpromotion"],
    dtype={"store_nbr": "category", "family": "category", "sales": "float32", "onpromotion": "uint32"},
    parse_dates=["date"],
)
store_sales["date"] = store_sales.date.dt.to_period("D")
store_sales = store_sales.set_index(["store_nbr", "family", "date"]).sort_index()

family_sales = (
    store_sales
    .groupby(["family", "date"], observed=True).mean()
    .unstack("family")
    .loc["2017"]
)
y = family_sales.loc[:, ("sales", "SCHOOL AND OFFICE SUPPLIES")].rename("sales")
onpromotion = family_sales.loc[:, ("onpromotion", "SCHOOL AND OFFICE SUPPLIES")].rename("onpromotion")

## 1. Deseasonalize

In [ ]:
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

dp = DeterministicProcess(
    index=y.index, constant=True, order=1, seasonal=True,
    additional_terms=[CalendarFourier(freq="ME", order=4)], drop=True,
)
X_time = dp.in_sample()

model = LinearRegression(fit_intercept=False).fit(X_time, y)
y_deseason = y - model.predict(X_time)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(11, 6))
y_deseason.plot(ax=ax1, title="Deseasonalized")
y.rolling(7, center=True).mean().plot(ax=ax2, title="7-day moving average: the cycles remain");

## 2. Choose lags: PACF + lag plots
The PACF shows how much *new* correlation each lag adds after the shorter lags. Keep lags whose bars fall outside the band.

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

plot_pacf(y_deseason, lags=8);
plot_lags(y_deseason, lags=8);

## 3. Leading indicator: promotions
Plot the target against lags **and leads** of the indicator.

In [ ]:
mask = onpromotion > 1   # ignore days with no promotion
plot_lags(onpromotion[mask], y_deseason[mask], lags=[3, 2, 1, 0, -1, -2, -3]);

## 4. Build features → fit → evaluate
Keep `shuffle=False` so the validation set comes after the training set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

X = pd.concat([
    X_time,
    make_lags(y_deseason, lags=1, name="sales"),
    make_lags(onpromotion, lags=1, name="promo"),
    onpromotion,
    make_leads(onpromotion, leads=1, name="promo"),
], axis=1).dropna()
y, X = y.align(X, join="inner")

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=30, shuffle=False)

model = LinearRegression(fit_intercept=False).fit(X_train, y_train)
y_fit = pd.Series(model.predict(X_train), index=X_train.index).clip(0.0)
y_pred = pd.Series(model.predict(X_valid), index=X_valid.index).clip(0.0)

print(f"Train RMSLE: {mean_squared_log_error(y_train, y_fit) ** 0.5:.4f}")
print(f"Valid RMSLE: {mean_squared_log_error(y_valid, y_pred) ** 0.5:.4f}")

ax = y.plot(**plot_params, alpha=0.5)
y_fit.plot(ax=ax, label="Fitted")
y_pred.plot(ax=ax, label="Forecast", color="C3")
ax.legend();

## 5. Rolling statistics
These are common in winning GBDT solutions. **Shift first, then roll**, so row *t* only sees data up to *t-1*.
Use a centered window only for series that are known in advance.

In [ ]:
y_lag = y.shift(1)

X_roll = pd.DataFrame({
    "mean_7": y_lag.rolling(7).mean(),
    "median_14": y_lag.rolling(14).median(),
    "std_7": y_lag.rolling(7).std(),
    "ewm_7": y_lag.ewm(span=7).mean(),                      # exponentially weighted
    "promo_7": onpromotion.rolling(7, center=True).sum(),   # centered: promos are planned
})
X_roll.dropna().head()

**Takeaways**
- Deseasonalize, then use the PACF and lag plots to pick the lags.
- Leading indicators help the model react before the target moves.
- Shift before rolling, or the features leak the future.